<a href="https://colab.research.google.com/github/diyanrahma/Retrieval-Augmented-Generation-Automatic-Fact-Checking-PubHealth-Dataset/blob/main/Distribusi_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/bigbio/pubhealth@refs/convert/parquet/pubhealth_bigbio_pairs/train/*.parquet",
    }
)["train"]

df = pd.DataFrame(ds)

df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
import re

def normalize_text(text):
    if pd.isnull(text):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"_", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["claim"] = df["text_1"].apply(normalize_text)
df["evidence"] = df["text_2"].apply(normalize_text)

df[["text_1", "claim"]].head()

In [ ]:
biomed_core = [
    "disease","diseases","illness","infection","infections","virus","viral",
    "bacteria","bacterial","fungus","fungal","pathogen","covid","covid-19",
    "influenza","flu","measles","mumps","hpv","hepatitis","ebola","malaria",
    "tuberculosis","tb","hiv","aids","cholera","dengue","pox","cancer","tumor",
    "carcinoma","leukemia","lymphoma","melanoma","diabetes","hypertension",
    "stroke","heart","cardio","cardiac","asthma","arthritis","alzheimer",
    "dementia","autism","obesity","metabolic","cholesterol","triglyceride",
    "immune","immunity","respiratory","digestive","circulatory","endocrine",
    "hormone","thyroid","insulin","kidney","renal","liver","hepatic","lungs",
    "brain","neuron","blood","cell","cells","tissue","organ","antibody",
    "antibodies","antigen","protein","enzyme","gene","genetic","genome","dna",
    "rna","mutation","microbiome","microorganism","biopsy","surgery","surgical",
    "scan","x-ray","ct-scan","mri","fertility","pregnancy","menopause",
    "contraception","breastfeeding","symptom","symptoms","fever","cough","pain",
    "inflammation","diagnosis","diagnose","screening","therapy","treatment",
    "antiviral","antibiotic","vaccine","vaccination","immunization","booster",
    "dose","dosing","medication","drug","drugs","medicine"
]
nutrition_keywords = [
    "food","foods","diet","eat","eating","drink","drinking","beverage","calorie",
    "calories","fat","fats","low fat","high fat","protein","carbohydrate","carbs",
    "sugar","salt","sodium","cholesterol","fiber","antioxidant","antioxidants",
    "organic","processed","unprocessed","whole grain","whole grains","gluten",
    "dairy","milk","cheese","yogurt","butter","cream","oil","olive oil",
    "coconut oil","vegetable oil","omega 3","omega-3","omega","fish oil",
    "nutrient","nutrients","vitamin","vitamins","mineral","minerals","supplement",
    "supplements","herbal","herb","spice","spices","fruit","fruits","vegetable",
    "vegetables","berries","tea","green tea","black tea","coffee","caffeine",
    "juice","smoothie","honey","ginger","garlic","turmeric","lemon","lime",
    "apple","banana","grape","orange","soy","soybean","beans","lentils",
    "nut","nuts","cocoa","chocolate","pepper","cinnamon","vinegar","apple cider"
]
wellness_keywords = [
    "health","healthy","healthier","wellness","wellbeing","well-being","immune",
    "immunity","boost","boosts","improves","improving","enhance","enhances",
    "prevents","prevent","prevention","reduce","reduces","lower","lowers",
    "digestion","digestive","gut","gut health","metabolism","metabolic",
    "energy","fatigue","tired","weight","weight loss","weight gain","fat burn",
    "burn fat","belly fat","body fat","fitness","exercise","workout","gym",
    "physical activity","movement","steps","walking","running","jogging",
    "yoga","meditation","strength","muscle","training","aerobic",
    "sleep","sleeping","rest","insomnia","stress","anxiety","mood",
    "mental health","focus","concentration","productivity",
]
alternative_keywords = [
    "essential oil","essential oils","herbal remedy","natural remedy",
    "home remedy","naturopathy","ayurveda","detox","cleanse","toxins",
    "purify","healing","cure","cures","remedy","remedies","antimicrobial",
    "antibacterial","antiviral","antifungal","antioxidant","anti-inflammatory",
    "anti-inflammatory","antiaging","anti-aging","homeopathy","holistic"
]
beauty_keywords = [
    "skin","skincare","acne","wrinkles","anti-aging","aging","hair","hair loss",
    "glow","glowing","moisture","hydration","moisturizing","sunburn","sunscreen",
    "collagen","elasticity","dark spots","pigmentation","dermatology"
]
general_health_keywords = [
    "heart health","brain health","bone health","joint","joints","lungs",
    "liver","kidney","blood flow","circulation","immune system","respiratory",
    "digestion","gut flora","microbiome","pain relief","headache","migraine",
    "cold","flu","sore throat","nausea","vomit","vomiting"
]
biomed_keywords = (
    biomed_core +
    nutrition_keywords +
    wellness_keywords +
    alternative_keywords +
    beauty_keywords +
    general_health_keywords
)
policy_keywords = [
    "government","policy","regulation","healthcare","insurance","mandate",
    "program","ministry","CDC","WHO","health system","funding","subsidy",
    "law","hospital policy","public health policy"
]

public_narrative_keywords = [
    "people","community","public","society","belief","myth","claim","spread",
    "misinformation","social media","population","public awareness","lifestyle"
]

def classify_topic(text):
    text = text.lower()

    if any(k in text for k in biomed_keywords):
        return "Biomedis / Medis"
    if any(k in text for k in policy_keywords):
        return "Kebijakan & Layanan Kesehatan Pemerintah"
    if any(k in text for k in public_narrative_keywords):
        return "Cerita / Narasi Kesehatan Masyarakat"
    return "Bukan Kesehatan"
df["topic"] = df["text_1"].apply(classify_topic)
print(df["topic"].value_counts())
print(df["label"].value_counts())
df.head()

In [ ]:
print("Jumlah per Label:", df["label"].value_counts().to_dict())

In [ ]:
import matplotlib.pyplot as plt

topic_counts = df["topic"].value_counts()

labels = topic_counts.index
values = topic_counts.values

plt.figure(figsize=(10, 5))
plt.bar(labels, values)

for i, v in enumerate(values):
    plt.text(i, v + max(values)*0.01, str(v), ha="center", va="bottom")

plt.title("Distribusi Topik PubHealth")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df_biomed = df[df["topic"] == "Biomedis / Medis"]
df_biomed_selected = df_biomed[["id", "claim", "evidence", "label", "topic"]]

print("Total data Biomedis / Medis:", len(df_biomed_selected))
df_biomed_selected.head(20)


In [ ]:
df_biomed = df[df["topic"] == "Biomedis / Medis"]
df_biomed_selected = df_biomed[["id", "claim", "evidence", "label", "topic"]]

df_biomed_selected.to_excel("pubhealth_biomed_selected.xlsx",
                            index=False,
                            engine="openpyxl")

print("File berhasil disimpan sebagai pubhealth_biomed_selected.xlsx")